# MDWD - Local Object Detection (YOLO26)

---

**Name:** Andrea Filiberto Lucas  
**Project:** MSc-MDWD-EUVIP26

---

YOLO26 training run with Weights & Biases tracking on MDWD

In [29]:
import os
from pathlib import Path

# Load local secrets from the repository-root .env file. The file is ignored by Git.
for candidate in [p / ".env" for p in [Path.cwd(), *Path.cwd().parents]]:
    if candidate.exists():
        for line in candidate.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip("\"'"))
        break

print("W&B credentials loaded successfully!" if os.getenv("WANDB_API_KEY") else "WANDB_API_KEY is not set; using existing W&B login (if available).")

W&B credentials loaded successfully!


In [30]:
import os
import re
import sys
import json
import time
import shutil
import random
import platform
import subprocess
import importlib.util
import importlib.metadata as importlib_metadata
from copy import deepcopy
from datetime import datetime
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

COLORS = {"green": "\033[92m", "red": "\033[91m", "yellow": "\033[93m", "reset": "\033[0m"}

def find_repo_root(start=None):
    start_path = Path(start or os.getcwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "Datasets").exists() and (candidate / "Scripts").exists() and (candidate / "Models").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing Datasets/, Scripts/ and Models/.")

ROOT_DIR = find_repo_root()
VERSIONS_DIR = ROOT_DIR / "Datasets" / "MDWD"
MODELS_DIR = ROOT_DIR / "Models"
DATASET_DIR = VERSIONS_DIR / "MDWD-YOLO26"
DATA_YAML_PATH = DATASET_DIR / "data.yaml"
VERSIONS_DIR.mkdir(exist_ok=True)

lib_file_path = ROOT_DIR / "Libraries" / "YOLO.json"
PACKAGE_RULES = {
    "ultralytics": {"min": "8.4.60"},
    "numpy": {"import": "numpy"},
    "opencv-python": {"import": "cv2"},
    "wandb": {"import": "wandb"},
    "PyYAML": {"import": "yaml"},
}

def parse_version(value):
    parts = re.findall(r"\d+", str(value))[:3]
    return tuple(int(part) for part in parts + ["0"] * (3 - len(parts)))

def package_version(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return None

def version_satisfies(version, rule):
    if version is None:
        return False
    parsed = parse_version(version)
    min_version = rule.get("min")
    max_exclusive = rule.get("max_exclusive")
    return not ((min_version and parsed < parse_version(min_version)) or (max_exclusive and parsed >= parse_version(max_exclusive)))

def spec_for(package_name, rule):
    spec = package_name
    if rule.get("min"):
        spec += f">={rule['min']}"
    if rule.get("max_exclusive"):
        spec += f"<{rule['max_exclusive']}"
    return spec

def check_and_install_libraries():
    # Legacy Libraries/YOLO.json pin list is optional; PACKAGE_RULES is the fallback.
    if lib_file_path.exists():
        with open(lib_file_path, "r", encoding="utf-8") as file:
            libraries = json.load(file)
    else:
        libraries = {}
    packages = dict(libraries)
    for package_name, rule in PACKAGE_RULES.items():
        packages.setdefault(package_name, rule.get("import", package_name))

    changed = False
    for package_name, import_name in packages.items():
        rule = PACKAGE_RULES.get(package_name, {})
        installed = importlib.util.find_spec(import_name) is not None
        current_version = package_version(package_name)
        if installed and version_satisfies(current_version, rule):
            version_text = f" ({current_version})" if current_version else ""
            print(f"[{COLORS['green']}OK{COLORS['reset']}] Library '{package_name}' is installed{version_text}.")
            continue
        spec = spec_for(package_name, rule)
        print(f"[{COLORS['yellow']}...{COLORS['reset']}] Installing/updating '{spec}'...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", spec])
        changed = True

    if changed:
        raise RuntimeError("Packages were installed or updated. Restart the Jupyter kernel, then rerun from the top.")

check_and_install_libraries()

import yaml
import torch
import pandas as pd  # type: ignore
import numpy as np  # type: ignore
from ultralytics import YOLO  # type: ignore

[OK] Library 'opencv-python' is installed (4.10.0.84).
[OK] Library 'matplotlib' is installed (3.10.6).
[OK] Library 'tqdm' is installed (4.67.1).
[OK] Library 'ultralytics' is installed (8.4.60).
[OK] Library 'torch' is installed (2.10.0.dev20251013+cu130).
[OK] Library 'wandb' is installed (0.27.0).
[OK] Library 'numpy' is installed (1.26.4).
[OK] Library 'PyYAML' is installed (6.0.2).


In [31]:
if DATASET_DIR.exists():
    print(f"[{COLORS['green']}OK{COLORS['reset']}] Dataset folder exists: {DATASET_DIR}")
else:
    if importlib.util.find_spec("roboflow") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "roboflow"])
        raise RuntimeError("Roboflow was installed. Restart the Jupyter kernel, then rerun from the top.")
    from roboflow import Roboflow  # type: ignore

    api_key = input("Please enter your Roboflow API key: ").strip()
    rf = Roboflow(api_key=api_key)
    project = rf.workspace("um-vbsez").project("maltese-garbage-bag-detection-xgodx")
    dataset = project.version(20).download("yolo26")
    downloaded_dir = Path(dataset.location).resolve()
    shutil.move(str(downloaded_dir), str(DATASET_DIR))
    print(f"[{COLORS['green']}OK{COLORS['reset']}] Dataset downloaded to: {DATASET_DIR}")

if not DATA_YAML_PATH.exists():
    raise FileNotFoundError(f"Dataset YAML not found: {DATA_YAML_PATH}")

[OK] Dataset folder exists: E:\1. UM-Staff\AICOM\AICOM-OD\Dataset-Versions\MDWD-YOLO26


## Hardware, Reproducibility, and Paths

In [32]:
RUNS_DIR = ROOT_DIR / "Results" / "MDWD-Runs" / "YOLO26-EUVIP"
RESULTS_DIR = ROOT_DIR / "Results" / "MDWD-Results" / "YOLO26"
BENCHMARK_RESULTS_DIR = RESULTS_DIR / "Model-Size-Comparison"
RUN_CONFIG_DIR = BENCHMARK_RESULTS_DIR / "run_configs"

for directory in [RUNS_DIR, BENCHMARK_RESULTS_DIR, RUN_CONFIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def determine_device(prefer_mps=True):
    if torch.cuda.is_available():
        device = "cuda"
        print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
    elif prefer_mps and torch.backends.mps.is_available():
        device = "mps"
        print("Using MPS (Apple Metal Performance Shaders).")
    else:
        device = "cpu"
        print("Defaulting to CPU. Training may be slow without GPU acceleration.")
    return device


def get_hardware_info(device=None):
    device = device or determine_device(prefer_mps=True)

    info = {
        "device": device,
        "platform": platform.platform(),
        "processor": platform.processor(),
        "cpu_count": os.cpu_count(),
        "python": sys.version.replace("\n", " "),
        "torch_version": getattr(torch, "__version__", None),
        "cuda_available": torch.cuda.is_available(),
        "mps_available": torch.backends.mps.is_available(),
        "mps_built": torch.backends.mps.is_built(),
    }

    if torch.cuda.is_available():
        info["cuda_version"] = torch.version.cuda
        info["cuda_device_count"] = torch.cuda.device_count()
        info["cuda_devices"] = []

        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            info["cuda_devices"].append({
                "index": idx,
                "name": props.name,
                "total_memory_gb": round(props.total_memory / (1024 ** 3), 2),
                "major": props.major,
                "minor": props.minor,
                "multi_processor_count": props.multi_processor_count,
            })

    return info


def default_workers(device):
    if device in {"mps", "cpu"}:
        return 0
    return min(8, max(2, (os.cpu_count() or 4) // 2))


def set_global_seed(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception as exc:
            print(f"Could not enable full deterministic algorithms: {exc}")

        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


def model_weights_path(filename):
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    path = (MODELS_DIR / filename).resolve()

    if path.exists():
        print(f"Using model weights from Models/: {path}")
        return str(path)

    # If an earlier notebook run downloaded weights beside the notebook, move them
    # into the canonical Models/ folder before falling back to an online download.
    stray_candidates = [Path.cwd() / filename, ROOT_DIR / "Jupyter Notebooks" / filename]
    for candidate in stray_candidates:
        candidate = candidate.resolve()
        if candidate.exists() and candidate != path:
            shutil.move(str(candidate), str(path))
            print(f"Moved stray model weights to Models/: {path}")
            return str(path)

    cwd = Path.cwd()
    try:
        os.chdir(MODELS_DIR)
        try:
            from ultralytics.utils.downloads import attempt_download_asset
            downloaded = Path(attempt_download_asset(filename)).resolve()
        except Exception:
            YOLO(filename)
            downloaded = (MODELS_DIR / filename).resolve()
    finally:
        os.chdir(cwd)

    if downloaded.exists() and downloaded.resolve() != path:
        shutil.move(str(downloaded), str(path))

    if not path.exists():
        raise FileNotFoundError(
            f"Model weights not found and could not be downloaded to root Models/: {path}"
        )

    print(f"Using model weights from Models/: {path}")
    return str(path)

DEVICE = determine_device(prefer_mps=True)
WORKERS = default_workers(DEVICE)
HARDWARE_INFO = get_hardware_info(DEVICE)

RUN_METADATA = {
    "experiment_name": "YOLO26-EUVIP-Model-Size-Comparison",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "root_dir": str(ROOT_DIR),
    "runs_dir": str(RUNS_DIR),
    "results_dir": str(RESULTS_DIR),
    "benchmark_results_dir": str(BENCHMARK_RESULTS_DIR),
    "run_config_dir": str(RUN_CONFIG_DIR),
    "device": DEVICE,
    "workers": WORKERS,
    "default_batch_size": 32,
    "hardware": HARDWARE_INFO,
}

hardware_config_path = RUN_CONFIG_DIR / "hardware_and_runtime_config.json"

with open(hardware_config_path, "w") as f:
    json.dump(RUN_METADATA, f, indent=2, default=str)

print(f"Training defaults -> device: {DEVICE}, workers: {WORKERS}")
print(f"Saved hardware/runtime config to: {hardware_config_path}")

Using CUDA GPU: NVIDIA GeForce RTX 4090
Training defaults -> device: cuda, workers: 8
Saved hardware/runtime config to: E:\1. UM-Staff\AICOM\AICOM-OD\Results\YOLO26\Model-Size-Comparison\run_configs\hardware_and_runtime_config.json


## Weights & Biases

In [33]:
WANDB_PROJECT = "MSc-MDWD-EUVIP26"
WANDB_ENTITY = os.getenv("WANDB_ENTITY", "mark-bugeja-university-of-malta")
WANDB_DEFAULT_MODE = os.getenv("WANDB_MODE", "online")
WANDB_AVAILABLE = importlib.util.find_spec("wandb") is not None
WANDB_RUN = None
WANDB_LOG_INTERVAL_STEPS = 100

if WANDB_AVAILABLE:
    import wandb  # type: ignore
else:
    wandb = None  # type: ignore
    print("W&B is not installed. Training can still run with W&B disabled.")

def configure_wandb_metrics(run):
    if run is None or wandb is None:
        return
    try:
        run.define_metric("trainer/global_step")
        run.define_metric("epoch", step_metric="trainer/global_step")
        for pattern in ["train/*", "val/*", "metrics/*", "lr/*", "final/*", "validation/*", "test/*", "runtime/*", "model/*"]:
            run.define_metric(pattern, step_metric="trainer/global_step")
    except Exception as exc:
        print(f"Could not configure W&B metric axes: {exc}")

def setup_wandb(config, mode=None):
    global WANDB_RUN
    if not WANDB_AVAILABLE or wandb is None:
        print("W&B unavailable; continuing without W&B logging.")
        return None

    requested_mode = str(mode or config.get("wandb", {}).get("mode") or WANDB_DEFAULT_MODE).lower()
    if requested_mode == "disabled":
        os.environ["WANDB_MODE"] = "disabled"
        return None

    try:
        if requested_mode == "online" and os.getenv("WANDB_API_KEY"):
            wandb.login(key=os.getenv("WANDB_API_KEY"), relogin=False, verify=True)
        elif requested_mode == "online":
            print("WANDB_MODE=online but WANDB_API_KEY is not set. Using existing W&B login if available.")

        if wandb.run is not None:
            wandb.run.finish()

        run = wandb.init(
            project=config.get("wandb", {}).get("project", WANDB_PROJECT),
            entity=WANDB_ENTITY,
            name=config["run_name"],
            group=config.get("wandb_group", "YOLO26-Model-Scale-Template"),
            config=config,
            tags=config.get("tags", []),
            notes=config.get("notes", ""),
            mode=requested_mode,
            resume="never",
            job_type="train",
            settings=wandb.Settings(),
        )
        WANDB_RUN = run
        configure_wandb_metrics(run)
        return run
    except Exception as exc:
        print(f"W&B online init failed ({exc}). Retrying in offline mode.")
        try:
            os.environ["WANDB_MODE"] = "offline"
            run = wandb.init(
                project=config.get("wandb", {}).get("project", WANDB_PROJECT),
                entity=WANDB_ENTITY,
                name=config["run_name"],
                group=config.get("wandb_group", "YOLO26-Model-Scale-Template"),
                config=config,
                tags=config.get("tags", []),
                notes=config.get("notes", ""),
                mode="offline",
                resume="never",
                job_type="train",
                settings=wandb.Settings(),
            )
            WANDB_RUN = run
            configure_wandb_metrics(run)
            return run
        except Exception as offline_exc:
            print(f"Offline W&B init also failed ({offline_exc}). Continuing without W&B.")
            return None

def attach_wandb_train_logger(model, run, log_interval_steps=WANDB_LOG_INTERVAL_STEPS):
    if run is None or wandb is None or not hasattr(model, "add_callback"):
        return
    log_interval_steps = int(log_interval_steps or WANDB_LOG_INTERVAL_STEPS)
    step_state = {"global_step": 0}

    def _as_float(value):
        try:
            if hasattr(value, "detach"):
                value = value.detach().mean()
            if hasattr(value, "item"):
                value = value.item()
            return float(value)
        except (TypeError, ValueError):
            return None

    def _base_payload(trainer):
        step_state["global_step"] += 1
        payload = {"trainer/global_step": step_state["global_step"]}
        epoch = getattr(trainer, "epoch", None)
        if epoch is not None:
            payload["epoch"] = int(epoch) + 1
        return payload

    def _add_losses(payload, trainer):
        loss_items = getattr(trainer, "loss_items", None)
        if loss_items is None:
            loss_items = getattr(trainer, "tloss", None)
        if loss_items is None:
            return
        try:
            values = list(loss_items)
        except TypeError:
            values = [loss_items]
        total = 0.0
        count = 0
        for name, value in zip(["box_loss", "cls_loss", "dfl_loss"], values):
            numeric = _as_float(value)
            if numeric is not None:
                payload[f"train/{name}"] = numeric
                total += numeric
                count += 1
        if count:
            payload["train/loss"] = total

    def _add_lr(payload, trainer):
        optimizer = getattr(trainer, "optimizer", None)
        for idx, group in enumerate(getattr(optimizer, "param_groups", []) if optimizer is not None else []):
            numeric = _as_float(group.get("lr"))
            if numeric is not None:
                payload[f"lr/pg{idx}"] = numeric
        if "lr/pg0" in payload:
            payload["lr"] = payload["lr/pg0"]

    def _csv_metrics_payload(trainer):
        save_dir = Path(getattr(trainer, "save_dir", ""))
        results_path = save_dir / "results.csv"
        if not results_path.exists():
            return {}
        try:
            df = pd.read_csv(results_path)
            if df.empty:
                return {}
            row = {str(key).strip(): value for key, value in df.iloc[-1].to_dict().items()}
            payload = {}
            for key, value in row.items():
                numeric = _as_float(value)
                if numeric is not None:
                    payload[key] = numeric
            canonical_metrics = {
                "metrics/precision": ["metrics/precision(B)", "metrics/precision"],
                "metrics/recall": ["metrics/recall(B)", "metrics/recall"],
                "metrics/mAP50": ["metrics/mAP50(B)", "metrics/mAP50"],
                "metrics/mAP50-95": ["metrics/mAP50-95(B)", "metrics/mAP50-95"],
                "val/box_loss": ["val/box_loss"],
                "val/cls_loss": ["val/cls_loss"],
                "val/dfl_loss": ["val/dfl_loss"],
            }
            for clean_name, candidates in canonical_metrics.items():
                for candidate in candidates:
                    if candidate in payload:
                        payload[clean_name] = payload[candidate]
                        break
            precision = payload.get("metrics/precision")
            recall = payload.get("metrics/recall")
            if precision is not None and recall is not None and (precision + recall) > 0:
                payload["metrics/F1"] = 2 * precision * recall / (precision + recall)
            if "time" in payload:
                payload["time/epoch_seconds"] = payload["time"]
            return payload
        except Exception as exc:
            print(f"W&B results.csv metric read skipped: {exc}")
            return {}

    def _log_train_batch(trainer):
        try:
            payload = _base_payload(trainer)
            if log_interval_steps <= 0 or payload["trainer/global_step"] % log_interval_steps != 0:
                return
            _add_losses(payload, trainer)
            _add_lr(payload, trainer)
            run.log(payload, commit=True)
        except Exception as exc:
            print(f"W&B step logging skipped: {exc}")

    def _log_epoch(trainer):
        try:
            payload = {"trainer/global_step": step_state["global_step"]}
            epoch = getattr(trainer, "epoch", None)
            if epoch is not None:
                payload["epoch"] = int(epoch) + 1
            metrics = getattr(trainer, "metrics", None) or {}
            for key, value in metrics.items() if isinstance(metrics, dict) else []:
                numeric = _as_float(value)
                if numeric is not None:
                    payload[key] = numeric
            payload.update(_csv_metrics_payload(trainer))
            _add_losses(payload, trainer)
            _add_lr(payload, trainer)
            run.log(payload, commit=True)
        except Exception as exc:
            print(f"W&B epoch logging skipped: {exc}")

    model.add_callback("on_train_batch_end", _log_train_batch)
    model.add_callback("on_fit_epoch_end", _log_epoch)
    print(f"Attached W&B train logger: losses every {log_interval_steps} steps; metrics every epoch.")

def finish_wandb_run(run=None):
    if WANDB_AVAILABLE and wandb is not None:
        active = run or wandb.run
        if active is not None:
            active.finish()
    globals()["WANDB_RUN"] = None

## Shared Training Template Configuration


In [34]:
ROBOFLOW_WORKSPACE = "um-vbsez"
ROBOFLOW_PROJECT = "maltese-garbage-bag-detection-xgodx"
ROBOFLOW_VERSION = 20
YOLO_EXPORT_FORMAT = "yolo26"

# Template control point: edit this list to train any set/order of YOLO26 model scales
# with the exact same dataset, hyperparameters, optimizer, augmentation, and logging settings.
YOLO26_MODEL_VARIANTS = ["yolo26n", "yolo26s", "yolo26m", "yolo26l"]
EXPERIMENT_PREFIX_TEMPLATE = "E{index:03d}"
EXPERIMENT_FAMILY = "YOLO26-Model-Scale-Template"

NO_ONLINE_AUGMENTATION_ARGS = {
    "hsv_h": 0.0, "hsv_s": 0.0, "hsv_v": 0.0,
    "degrees": 0.0, "translate": 0.0, "scale": 0.0, "shear": 0.0, "perspective": 0.0,
    "flipud": 0.0, "fliplr": 0.0, "bgr": 0.0,
    "mosaic": 0.0, "mixup": 0.0, "cutmix": 0.0, "copy_paste": 0.0,
    "erasing": 0.0, "auto_augment": None, "close_mosaic": 0,
}

base_training_config = {
    "experiment_number": 6,
    "experiment_type": "multi-model-template",
    "configuration_source": "shared_yolo26_rfv20_img640_b32_e100_adamw_lr0p001",
    "dataset_path": str(DATA_YAML_PATH),
    "dataset_dir": str(DATASET_DIR),
    "roboflow": {
        "workspace": ROBOFLOW_WORKSPACE,
        "project": ROBOFLOW_PROJECT,
        "version": ROBOFLOW_VERSION,
        "export_format": YOLO_EXPORT_FORMAT,
        "dataset_augmentation": "pre-augmented export; online YOLO augmentation disabled",
    },
    "epochs": 100,
    "imgsz": 640,
    "batch": 32,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "weight_decay": 0.0005,
    "momentum": 0.937,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "cos_lr": False,
    "patience": 10,
    "seed": 42,
    "deterministic": True,
    "conf": 0.001,
    "iou": 0.7,
    "device": DEVICE,
    "workers": WORKERS,
    "project_dir": str(RUNS_DIR),
    "save_period": -1,
    "plots": True,
    "pretrained": True,
    "exist_ok": False,
    "online_augmentation": "disabled",
    "train_overrides": deepcopy(NO_ONLINE_AUGMENTATION_ARGS),
    "wandb_group": EXPERIMENT_FAMILY,
    "tags": ["yolo26", "multi-model-template", "model-scale-benchmark", "adamw", "img640", "rfv20"],
    "notes": "Template notebook: train multiple YOLO26 model variants with identical training settings by editing YOLO26_MODEL_VARIANTS.",
    "wandb": {"project": WANDB_PROJECT, "mode": WANDB_DEFAULT_MODE, "log_interval_steps": WANDB_LOG_INTERVAL_STEPS, "use_ultralytics_callback": True},
    "hardware": {
        "device": DEVICE,
        "workers": WORKERS,
        "cuda_available": HARDWARE_INFO.get("cuda_available"),
        "gpu": HARDWARE_INFO.get("cuda_devices", [{}])[0].get("name") if HARDWARE_INFO.get("cuda_devices") else None,
        "gpu_vram_gb": HARDWARE_INFO.get("cuda_devices", [{}])[0].get("total_memory_gb") if HARDWARE_INFO.get("cuda_devices") else None,
    },
}

def safe_run_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(text)).strip("-")

base_training_config["created_at"] = datetime.now().isoformat(timespec="seconds")
template_config_path = RUN_CONFIG_DIR / "base_training_template.json"
template_config_path.write_text(json.dumps(base_training_config, indent=2, default=str), encoding="utf-8")

print("Shared training template saved to:", template_config_path)
print("Model variants to train:", YOLO26_MODEL_VARIANTS)
print(json.dumps(base_training_config, indent=2, default=str))

Shared training template saved to: E:\1. UM-Staff\AICOM\AICOM-OD\Results\YOLO26\Model-Size-Comparison\run_configs\base_training_template.json
Model variants to train: ['yolo26n', 'yolo26s', 'yolo26m', 'yolo26l']
{
  "experiment_number": 6,
  "experiment_type": "multi-model-template",
  "configuration_source": "shared_yolo26_rfv20_img640_b32_e100_adamw_lr0p001",
  "dataset_path": "E:\\1. UM-Staff\\AICOM\\AICOM-OD\\Dataset-Versions\\MDWD-YOLO26\\data.yaml",
  "dataset_dir": "E:\\1. UM-Staff\\AICOM\\AICOM-OD\\Dataset-Versions\\MDWD-YOLO26",
  "roboflow": {
    "workspace": "um-vbsez",
    "project": "maltese-garbage-bag-detection-xgodx",
    "version": 20,
    "export_format": "yolo26",
    "dataset_augmentation": "pre-augmented export; online YOLO augmentation disabled"
  },
  "epochs": 100,
  "imgsz": 640,
  "batch": 32,
  "optimizer": "AdamW",
  "lr0": 0.001,
  "lrf": 0.01,
  "weight_decay": 0.0005,
  "momentum": 0.937,
  "warmup_epochs": 3.0,
  "warmup_momentum": 0.8,
  "warmup_bias_l

## Training Helpers

In [35]:
def read_results_csv(save_dir):
    path = Path(save_dir) / "results.csv"
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def metric_value(metrics, candidates):
    for key in candidates:
        if key in metrics and pd.notna(metrics[key]):
            return metrics[key]
    return None

def summarize_metrics(save_dir):
    df = read_results_csv(save_dir)
    if df.empty:
        return {}
    df.columns = [str(column).strip() for column in df.columns]
    final = df.iloc[-1].to_dict()
    map_column = next((column for column in ["metrics/mAP50-95(B)", "metrics/mAP50-95"] if column in df), None)
    best_index = df[map_column].astype(float).idxmax() if map_column else df.index[-1]
    best = df.loc[best_index].to_dict()
    summary = {
        "precision": metric_value(best, ["metrics/precision(B)", "metrics/precision"]),
        "recall": metric_value(best, ["metrics/recall(B)", "metrics/recall"]),
        "mAP50": metric_value(best, ["metrics/mAP50(B)", "metrics/mAP50"]),
        "mAP50-95": metric_value(best, ["metrics/mAP50-95(B)", "metrics/mAP50-95"]),
        "best_epoch": int(best.get("epoch", best_index)) + 1,
        "completed_epochs": len(df),
        "train_box_loss": metric_value(final, ["train/box_loss"]),
        "train_cls_loss": metric_value(final, ["train/cls_loss"]),
        "train_dfl_loss": metric_value(final, ["train/dfl_loss"]),
        "val_box_loss": metric_value(best, ["val/box_loss"]),
        "val_cls_loss": metric_value(best, ["val/cls_loss"]),
        "val_dfl_loss": metric_value(best, ["val/dfl_loss"]),
        "lr": metric_value(final, ["lr/pg0", "lr/pg1", "lr/pg2"]),
    }
    precision, recall = summary.get("precision"), summary.get("recall")
    summary["F1"] = 2 * precision * recall / (precision + recall) if precision is not None and recall is not None and (precision + recall) > 0 else None
    return summary

def log_dataset_metadata_to_wandb(run, config):
    if run is None or wandb is None:
        return
    try:
        artifact = wandb.Artifact(
            name=f"roboflow-{config['roboflow']['project']}-v{config['roboflow']['version']}",
            type="dataset",
            metadata={"dataset_path": config["dataset_path"], "dataset_dir": config["dataset_dir"], **config["roboflow"]},
        )
        artifact.add_file(config["dataset_path"])
        run.log_artifact(artifact)
    except Exception as exc:
        print(f"Could not log dataset metadata artifact: {exc}")

def log_artifacts_to_wandb(run, config, save_dir):
    if run is None or wandb is None:
        return
    save_dir = Path(save_dir)
    try:
        for filename in ["results.csv", "args.yaml"]:
            file_path = save_dir / filename
            if file_path.exists():
                wandb.save(str(file_path))
        for pattern in ["confusion_matrix*.png", "PR_curve*.png", "P_curve*.png", "R_curve*.png", "F1_curve*.png", "results.png", "labels*.jpg", "train_batch*.jpg", "val_batch*.jpg"]:
            for file_path in save_dir.glob(pattern):
                wandb.save(str(file_path))
        model_artifact = wandb.Artifact(
            name=f"{config['run_name']}-weights",
            type="model",
            metadata={"run_name": config["run_name"], "mAP50-95": config.get("final_metrics", {}).get("mAP50-95")},
        )
        added = False
        for weight_name in ["best.pt", "last.pt"]:
            weight_path = save_dir / "weights" / weight_name
            if weight_path.exists():
                model_artifact.add_file(str(weight_path), name=weight_name)
                added = True
        if added:
            run.log_artifact(model_artifact)
        final_payload = {f"final/{key}": value for key, value in config.get("final_metrics", {}).items() if value is not None}
        if config.get("train_seconds") is not None:
            final_payload["final/train_seconds"] = config["train_seconds"]
            final_payload["final/train_minutes"] = config["train_seconds"] / 60
        if final_payload:
            wandb.log(final_payload)
    except Exception as exc:
        print(f"Could not log W&B artifacts: {exc}")

def validate_yolo26_experiment(model, config, split="val"):
    return model.val(
        data=config["dataset_path"],
        split=split,
        imgsz=config["imgsz"],
        batch=config["batch"],
        conf=config["conf"],
        iou=config["iou"],
        device=config["device"],
        workers=config["workers"],
        plots=True,
    )

def train_yolo26_experiment(config, finish_run=True):
    config = deepcopy(config)
    intended_dir = Path(config["project_dir"]) / config["run_name"]
    if intended_dir.exists():
        raise RuntimeError(f"Run directory already exists and will not be overwritten: {intended_dir}")

    set_global_seed(config.get("seed", 42), deterministic=config.get("deterministic", True))
    run = setup_wandb(config)
    log_dataset_metadata_to_wandb(run, config)
    model = YOLO(model_weights_path(config["model_checkpoint"]))

    if run is not None:
        attach_wandb_train_logger(model, run, config.get("wandb", {}).get("log_interval_steps", WANDB_LOG_INTERVAL_STEPS))
        if config.get("wandb", {}).get("use_ultralytics_callback", False):
            try:
                from wandb.integration.ultralytics import add_wandb_callback  # type: ignore
                add_wandb_callback(model, enable_model_checkpointing=False)
            except Exception as exc:
                print(f"W&B Ultralytics callback unavailable; continuing with custom/manual logging: {exc}")

    train_args = {
        "data": config["dataset_path"], "epochs": config["epochs"], "imgsz": config["imgsz"], "batch": config["batch"],
        "name": config["run_name"], "project": config["project_dir"], "pretrained": config.get("pretrained", True),
        "plots": config.get("plots", True), "device": config["device"], "workers": config["workers"],
        "optimizer": config["optimizer"], "lr0": config["lr0"], "lrf": config["lrf"],
        "weight_decay": config["weight_decay"], "momentum": config["momentum"],
        "warmup_epochs": config["warmup_epochs"], "warmup_momentum": config["warmup_momentum"], "warmup_bias_lr": config["warmup_bias_lr"],
        "box": config["box"], "cls": config["cls"], "dfl": config["dfl"], "cos_lr": config["cos_lr"],
        "patience": config["patience"], "seed": config["seed"], "deterministic": config["deterministic"],
        "save_period": config.get("save_period", -1), "exist_ok": False, **deepcopy(NO_ONLINE_AUGMENTATION_ARGS),
    }

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start_time = time.time()
    try:
        train_result = model.train(**train_args)
        train_seconds = time.time() - start_time
        save_dir = Path(getattr(model.trainer, "save_dir", intended_dir))
        config["save_dir"] = str(save_dir)
        config["train_seconds"] = train_seconds
        config["gpu_memory_peak_allocated_gb"] = round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3) if torch.cuda.is_available() else None
        config["gpu_memory_peak_reserved_gb"] = round(torch.cuda.max_memory_reserved() / (1024 ** 3), 3) if torch.cuda.is_available() else None
        config["best_weights"] = str(save_dir / "weights" / "best.pt")
        config["last_weights"] = str(save_dir / "weights" / "last.pt")
        val_result = validate_yolo26_experiment(model, config)
        config["final_metrics"] = summarize_metrics(save_dir)
        config_path = RUN_CONFIG_DIR / f"{config['run_name']}.json"
        config_path.write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")
        log_artifacts_to_wandb(run, config, save_dir)

        row = {
            "run_name": config["run_name"], "status": "completed", "model_architecture": config["model_architecture"],
            "model_checkpoint": config["model_checkpoint"], "imgsz": config["imgsz"], "batch": config["batch"],
            "epochs": config["epochs"], "optimizer": config["optimizer"], "lr0": config["lr0"], "lrf": config["lrf"],
            "weight_decay": config["weight_decay"], "momentum": config["momentum"], "warmup_epochs": config["warmup_epochs"],
            "online_augmentation": "disabled", "train_seconds": round(train_seconds, 2),
            "gpu_memory_peak_allocated_gb": config["gpu_memory_peak_allocated_gb"],
            "gpu_memory_peak_reserved_gb": config["gpu_memory_peak_reserved_gb"],
            "best_weights": config["best_weights"], "last_weights": config["last_weights"],
            "wandb_url": getattr(run, "url", None) if run is not None else None, **config["final_metrics"],
        }
        summary_path = BENCHMARK_RESULTS_DIR / "yolo26_model_run_summary.csv"
        pd.DataFrame([row]).to_csv(summary_path, index=False)
        with open(save_dir / "pipeline_complete.json", "w", encoding="utf-8") as file:
            json.dump(row, file, indent=2, default=str)
        print(f"Run summary exported: {summary_path}")
        return row, model, train_result, val_result
    finally:
        if finish_run:
            finish_wandb_run(run)

## Run Multi-Model Template


In [ ]:
RUN_TRAINING = True

MODEL_RUN_CONFIG_DIR = RUN_CONFIG_DIR
for directory in [BENCHMARK_RESULTS_DIR, MODEL_RUN_CONFIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if importlib.util.find_spec("psutil") is not None:
    try:
        import psutil  # type: ignore
        vm = psutil.virtual_memory()
        HARDWARE_INFO.update({
            "cpu_count_physical": psutil.cpu_count(logical=False),
            "cpu_count_logical": psutil.cpu_count(logical=True),
            "ram_total_gb": round(vm.total / (1024 ** 3), 2),
            "ram_available_gb": round(vm.available / (1024 ** 3), 2),
        })
    except Exception as exc:
        HARDWARE_INFO["psutil_warning"] = str(exc)

def make_model_run_config(base_config, model_variant, experiment_index):
    config = deepcopy(base_config)
    config["experiment_type"] = "model-scale-benchmark"
    config["model_architecture"] = model_variant
    config["model_checkpoint"] = f"{model_variant}.pt"
    config["benchmark_variant"] = model_variant
    experiment_prefix = EXPERIMENT_PREFIX_TEMPLATE.format(index=experiment_index)
    config["experiment_prefix"] = experiment_prefix
    config["experiment_index"] = experiment_index
    config["run_name"] = safe_run_name(f"{experiment_prefix}_{model_variant}_rfv20_img640_b32_e100_adamw_lr0p001")
    config["wandb_group"] = EXPERIMENT_FAMILY
    config["tags"] = sorted(set(config.get("tags", []) + [model_variant, "model-scale-benchmark", "scale-ablation"]))
    config["notes"] = f"Template run for {model_variant}: identical shared training settings, only the model checkpoint changes."
    config["hardware"] = HARDWARE_INFO
    config["created_at"] = datetime.now().isoformat(timespec="seconds")
    return config

model_run_configs = [
    make_model_run_config(base_training_config, variant, index)
    for index, variant in enumerate(YOLO26_MODEL_VARIANTS, start=1)
]
for config in model_run_configs:
    (MODEL_RUN_CONFIG_DIR / f"{config['run_name']}.json").write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")

def dataset_metadata(config):
    metadata = {"dataset_path": config["dataset_path"], "dataset_dir": config["dataset_dir"], **config["roboflow"]}
    try:
        data = yaml.safe_load(Path(config["dataset_path"]).read_text(encoding="utf-8")) or {}
        root = Path(data.get("path", Path(config["dataset_path"]).parent))
        if not root.is_absolute():
            root = (Path(config["dataset_path"]).parent / root).resolve()
        metadata.update({"yaml": str(config["dataset_path"]), "root": str(root), "class_names": data.get("names", []), "class_count": len(data.get("names", []))})
        for split in ["train", "val", "test"]:
            split_value = data.get(split)
            if not split_value:
                metadata[f"{split}_images"] = None
                continue
            split_path = Path(split_value)
            if not split_path.is_absolute():
                split_path = root / split_path
            count = 0
            for pattern in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff", "*.webp"]:
                count += len(list(split_path.rglob(pattern))) if split_path.exists() else 0
            metadata[f"{split}_path"] = str(split_path)
            metadata[f"{split}_images"] = count
    except Exception as exc:
        metadata["warning"] = repr(exc)
    return metadata

def alert_wandb(run, title, text, level="INFO"):
    if run is None or wandb is None:
        print(f"W&B alert skipped: {title} - {text}")
        return
    try:
        wandb.alert(title=title, text=text, level=getattr(wandb.AlertLevel, level, wandb.AlertLevel.INFO))
    except Exception as exc:
        print(f"Could not send W&B alert: {exc}")

def log_event(run, name, payload=None):
    payload = payload or {}
    print(f"[{datetime.now().isoformat(timespec='seconds')}] {name}: {json.dumps(payload, default=str)}")
    if run is not None and wandb is not None:
        try:
            run.summary["last_event"] = {"name": name, **payload}
        except Exception as exc:
            print(f"Could not update W&B event summary {name}: {exc}")

def yolo_result_metrics(result, prefix):
    metrics = {}
    box = getattr(result, "box", None)
    for target, attrs in {"precision": ["mp", "precision"], "recall": ["mr", "recall"], "mAP50": ["map50"], "mAP50-95": ["map"]}.items():
        for attr in attrs:
            try:
                value = getattr(box, attr, None) if box is not None else None
                value = value.item() if hasattr(value, "item") else value
                if value is not None and pd.notna(value):
                    metrics[f"{prefix}/{target}"] = float(value)
                    break
            except Exception:
                pass
    precision, recall = metrics.get(f"{prefix}/precision"), metrics.get(f"{prefix}/recall")
    metrics[f"{prefix}/F1"] = 2 * precision * recall / (precision + recall) if precision is not None and recall is not None and (precision + recall) > 0 else None
    for key, value in (getattr(result, "speed", None) or {}).items():
        try:
            metrics[f"{prefix}/speed_{key}_ms"] = float(value)
        except Exception:
            pass
    metrics[f"{prefix}/inference_time_ms"] = metrics.get(f"{prefix}/speed_inference_ms")
    return metrics

def model_parameters(model):
    try:
        return int(sum(parameter.numel() for parameter in model.model.parameters()))
    except Exception:
        return None

def file_size_mb(path):
    path = Path(path)
    return round(path.stat().st_size / (1024 ** 2), 2) if path.exists() else None

def upload_benchmark_artifacts(run, config, output_dirs, csv_paths=None):
    if run is None or wandb is None:
        return
    try:
        artifact = wandb.Artifact(f"{config['run_name']}-benchmark-outputs", type="benchmark-outputs", metadata={"model_variant": config["model_architecture"]})
        added = False
        for directory in [Path(item) for item in output_dirs if item]:
            if not directory.exists():
                continue
            for filename in ["results.csv", "args.yaml", "pipeline_complete.json"]:
                file_path = directory / filename
                if file_path.exists():
                    artifact.add_file(str(file_path), name=f"{directory.name}/{filename}")
                    wandb.save(str(file_path))
                    added = True
            for pattern in ["confusion_matrix*.png", "PR_curve*.png", "P_curve*.png", "R_curve*.png", "F1_curve*.png", "results.png", "labels*.jpg", "train_batch*.jpg", "val_batch*.jpg"]:
                for file_path in directory.glob(pattern):
                    artifact.add_file(str(file_path), name=f"{directory.name}/{file_path.name}")
                    wandb.save(str(file_path))
                    added = True
        for csv_path in [Path(item) for item in (csv_paths or []) if item]:
            if csv_path.exists():
                artifact.add_file(str(csv_path), name=csv_path.name)
                wandb.save(str(csv_path))
                added = True
        if added:
            run.log_artifact(artifact)

        weights = wandb.Artifact(f"{config['run_name']}-weights", type="model", metadata={"model_variant": config["model_architecture"]})
        added_weights = False
        for weight_name in ["best.pt", "last.pt"]:
            weight_path = Path(config.get("save_dir", "")) / "weights" / weight_name
            if weight_path.exists():
                weights.add_file(str(weight_path), name=weight_name)
                added_weights = True
        if added_weights:
            run.log_artifact(weights)
    except Exception as exc:
        print(f"Could not upload benchmark artifacts: {exc}")

def train_yolo26_benchmark_run(config):
    config = deepcopy(config)
    intended_dir = Path(config["project_dir"]) / config["run_name"]
    run = None
    model = None
    start_time = time.time()
    try:
        if intended_dir.exists():
            raise RuntimeError(f"Run directory already exists and will not be overwritten: {intended_dir}")
        set_global_seed(config.get("seed", 42), deterministic=config.get("deterministic", True))
        run = setup_wandb(config)
        data_info = dataset_metadata(config)
        print("Dataset information:")
        print(json.dumps(data_info, indent=2, default=str))
        if run is not None:
            run.config.update({"dataset_info": data_info}, allow_val_change=True)
        log_dataset_metadata_to_wandb(run, config)

        log_event(run, "training_start", {"model_variant": config["model_architecture"], "checkpoint": config["model_checkpoint"]})
        model = YOLO(model_weights_path(config["model_checkpoint"]))
        config["parameter_count_initial"] = model_parameters(model)
        if run is not None:
            attach_wandb_train_logger(model, run, config.get("wandb", {}).get("log_interval_steps", WANDB_LOG_INTERVAL_STEPS))
            if config.get("wandb", {}).get("use_ultralytics_callback", False):
                try:
                    from wandb.integration.ultralytics import add_wandb_callback  # type: ignore
                    add_wandb_callback(model, enable_model_checkpointing=False)
                except Exception as exc:
                    log_event(run, "warning", {"message": f"W&B Ultralytics callback unavailable: {exc}"})

        train_args = {
            "data": config["dataset_path"], "epochs": config["epochs"], "imgsz": config["imgsz"], "batch": config["batch"],
            "name": config["run_name"], "project": config["project_dir"], "pretrained": config.get("pretrained", True),
            "plots": config.get("plots", True), "device": config["device"], "workers": config["workers"],
            "optimizer": config["optimizer"], "lr0": config["lr0"], "lrf": config["lrf"],
            "weight_decay": config["weight_decay"], "momentum": config["momentum"],
            "warmup_epochs": config["warmup_epochs"], "warmup_momentum": config["warmup_momentum"], "warmup_bias_lr": config["warmup_bias_lr"],
            "box": config["box"], "cls": config["cls"], "dfl": config["dfl"], "cos_lr": config["cos_lr"],
            "patience": config["patience"], "seed": config["seed"], "deterministic": config["deterministic"],
            "save_period": config.get("save_period", -1), "exist_ok": False, **deepcopy(NO_ONLINE_AUGMENTATION_ARGS),
        }
        if run is not None:
            run.config.update({"train_args": train_args}, allow_val_change=True)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        train_result = model.train(**train_args)
        train_seconds = time.time() - start_time
        save_dir = Path(getattr(model.trainer, "save_dir", intended_dir))
        config.update({
            "save_dir": str(save_dir),
            "train_seconds": train_seconds,
            "best_weights": str(save_dir / "weights" / "best.pt"),
            "last_weights": str(save_dir / "weights" / "last.pt"),
            "gpu_memory_peak_allocated_gb": round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3) if torch.cuda.is_available() else None,
            "gpu_memory_peak_reserved_gb": round(torch.cuda.max_memory_reserved() / (1024 ** 3), 3) if torch.cuda.is_available() else None,
        })
        log_event(run, "training_end", {"train_seconds": round(train_seconds, 2), "best_weights": config["best_weights"], "last_weights": config["last_weights"]})

        best_model = YOLO(config["best_weights"]) if Path(config["best_weights"]).exists() else model
        config["parameter_count"] = model_parameters(best_model)
        config["model_size_mb"] = file_size_mb(config["best_weights"])

        log_event(run, "validation_start", {"split": "val", "weights": config["best_weights"]})
        val_result = validate_yolo26_experiment(best_model, config, split="val")
        val_metrics = yolo_result_metrics(val_result, "validation")
        val_save_attr = getattr(val_result, "save_dir", None)
        val_dir = Path(val_save_attr) if val_save_attr else None
        log_event(run, "validation_end", {key.replace("validation/", ""): value for key, value in val_metrics.items()})

        log_event(run, "test_evaluation_start", {"split": "test", "weights": config["best_weights"]})
        test_result = validate_yolo26_experiment(best_model, config, split="test")
        test_metrics = yolo_result_metrics(test_result, "test")
        test_save_attr = getattr(test_result, "save_dir", None)
        test_dir = Path(test_save_attr) if test_save_attr else None
        log_event(run, "test_evaluation_end", {key.replace("test/", ""): value for key, value in test_metrics.items()})

        config["final_metrics"] = summarize_metrics(save_dir)
        (MODEL_RUN_CONFIG_DIR / f"{config['run_name']}.json").write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")
        row = {
            "model_variant": config["model_architecture"],
            "parameter_count": config["parameter_count"],
            "model_size_mb": config["model_size_mb"],
            "mAP@50": val_metrics.get("validation/mAP50", config["final_metrics"].get("mAP50")),
            "mAP@50-95": val_metrics.get("validation/mAP50-95", config["final_metrics"].get("mAP50-95")),
            "precision": val_metrics.get("validation/precision", config["final_metrics"].get("precision")),
            "recall": val_metrics.get("validation/recall", config["final_metrics"].get("recall")),
            "F1 Score": val_metrics.get("validation/F1", config["final_metrics"].get("F1")),
            "validation_inference_time_ms": val_metrics.get("validation/inference_time_ms"),
            "test_inference_time_ms": test_metrics.get("test/inference_time_ms"),
            "total_training_time_seconds": round(train_seconds, 2),
            "test_mAP@50": test_metrics.get("test/mAP50"),
            "test_mAP@50-95": test_metrics.get("test/mAP50-95"),
            "test_precision": test_metrics.get("test/precision"),
            "test_recall": test_metrics.get("test/recall"),
            "test_F1 Score": test_metrics.get("test/F1"),
            "best_weights": config["best_weights"],
            "last_weights": config["last_weights"],
            "run_name": config["run_name"],
            "status": "completed",
            "wandb_url": getattr(run, "url", None) if run is not None else None,
        }
        with open(save_dir / "pipeline_complete.json", "w", encoding="utf-8") as file:
            json.dump(row, file, indent=2, default=str)
        if run is not None:
            run.log({**val_metrics, **test_metrics, "runtime/train_seconds": train_seconds, "model/parameters": config["parameter_count"] or 0, "model/size_mb": config["model_size_mb"] or 0})
            for key, value in row.items():
                if isinstance(value, (int, float)) and pd.notna(value):
                    run.summary[key] = value
            run.summary["best_weights"] = config["best_weights"]
            run.summary["last_weights"] = config["last_weights"]
        upload_benchmark_artifacts(run, config, [save_dir, val_dir, test_dir])
        log_event(run, "model_summary", row)
        alert_wandb(run, f"{config['model_architecture']} benchmark completed", f"Validation mAP@50={row['mAP@50']}, mAP@50-95={row['mAP@50-95']}, F1={row['F1 Score']}; runtime={round(train_seconds / 60, 2)} min; status=success", "INFO")
        return row
    except Exception as exc:
        train_seconds = time.time() - start_time
        row = {"model_variant": config.get("model_architecture"), "run_name": config.get("run_name"), "status": "failed", "error": repr(exc), "total_training_time_seconds": round(train_seconds, 2)}
        print(f"Benchmark failed for {config.get('model_architecture')}: {exc}")
        log_event(run, "exception", row)
        if run is not None:
            run.summary["status"] = "failed"
            run.summary["error"] = repr(exc)
        alert_wandb(run, f"{config.get('model_architecture')} benchmark failed", f"status=failure; runtime={round(train_seconds / 60, 2)} min; error={repr(exc)}", "ERROR")
        return row
    finally:
        finish_wandb_run(run)

def log_model_run_summary(rows, summary_path):
    if wandb is None or not rows:
        return
    summary_config = {
        "run_name": safe_run_name(f"E006_yolo26_scale_benchmark_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}"),
        "wandb_group": "YOLO26-Model-Scale-Template",
        "tags": ["yolo26", "model-scale-benchmark", "summary"],
        "wandb": {"project": WANDB_PROJECT, "mode": WANDB_DEFAULT_MODE},
    }
    run = setup_wandb(summary_config)
    try:
        if run is None:
            return
        df = pd.DataFrame(rows)
        run.log({"benchmark/comparison_table": wandb.Table(dataframe=df)})
        artifact = wandb.Artifact("yolo26-multi-model-template-comparison", type="benchmark-summary", metadata={"rows": len(df)})
        artifact.add_file(str(summary_path), name=Path(summary_path).name)
        run.log_artifact(artifact)
        run.summary["summary_csv"] = str(summary_path)
    finally:
        finish_wandb_run(run)

def run_yolo26_multi_model_template(configs):
    rows = []
    for index, config in enumerate(configs, start=1):
        print(f"\n=== YOLO26 model-scale benchmark {index}/{len(configs)}: {config['model_architecture']} ===")
        row = train_yolo26_benchmark_run(config)
        rows.append(row)
        partial_path = BENCHMARK_RESULTS_DIR / "yolo26_multi_model_template_partial.csv"
        pd.DataFrame(rows).to_csv(partial_path, index=False)
        print(f"Partial benchmark results exported: {partial_path}")

    comparison_columns = [
        "model_variant", "parameter_count", "model_size_mb", "mAP@50", "mAP@50-95", "precision", "recall", "F1 Score",
        "validation_inference_time_ms", "test_inference_time_ms", "total_training_time_seconds",
    ]
    df = pd.DataFrame(rows)
    for column in comparison_columns:
        if column not in df.columns:
            df[column] = None
    comparison_df = df[comparison_columns + [column for column in df.columns if column not in comparison_columns]]
    summary_path = BENCHMARK_RESULTS_DIR / "yolo26_multi_model_template_summary.csv"
    comparison_df.to_csv(summary_path, index=False)
    print(f"Consolidated benchmark summary exported: {summary_path}")
    log_model_run_summary(comparison_df.to_dict(orient="records"), summary_path)
    return comparison_df

if RUN_TRAINING:
    model_run_summary_df = run_yolo26_multi_model_template(model_run_configs)
    display(model_run_summary_df)
else:
    print("RUN_TRAINING is False. Set it to True to start the YOLO26 model-scale benchmark.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\fridge\_netrc



=== YOLO26 model-scale benchmark 1/4: yolo26n ===


Dataset information:
{
  "dataset_path": "E:\\1. UM-Staff\\AICOM\\AICOM-OD\\Dataset-Versions\\MDWD-YOLO26\\data.yaml",
  "dataset_dir": "E:\\1. UM-Staff\\AICOM\\AICOM-OD\\Dataset-Versions\\MDWD-YOLO26",
  "workspace": "um-vbsez",
  "project": "maltese-garbage-bag-detection-xgodx",
  "version": 20,
  "export_format": "yolo26",
  "dataset_augmentation": "pre-augmented export; online YOLO augmentation disabled",
  "yaml": "E:\\1. UM-Staff\\AICOM\\AICOM-OD\\Dataset-Versions\\MDWD-YOLO26\\data.yaml",
  "root": "E:\\1. UM-Staff\\AICOM\\AICOM-OD\\Dataset-Versions\\MDWD-YOLO26",
  "class_names": [
    "Mixed Waste",
    "Orange CMD",
    "Organic Waste",
    "Other Waste",
    "Recyclable Material"
  ],
  "class_count": 5,
  "train_path": "E:\\1. UM-Staff\\AICOM\\AICOM-OD\\Dataset-Versions\\MDWD-YOLO26\\..\\train\\images",
  "train_images": 0,
  "val_path": "E:\\1. UM-Staff\\AICOM\\AICOM-OD\\Dataset-Versions\\MDWD-YOLO26\\..\\valid\\images",
  "val_images": 0,
  "test_path": "E:\\1. UM-Staff\\

wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Copied 1 file (cross-volume or links unavailable). Downgrading policy to 'now' for those files because live updates won't propagate from the originals. Re-run wandb.save to resync, or place your run directory on the same drive to enable hardlinks.
wandb: WARNING Copied 1 file (cross-volume or links unavailable). Downgrading policy to 'now' for those files because live updates won't propagate from the originals. Re-run wandb.save to resync, or place your run directory on the same drive to enable hardlinks.
wandb: WARNING Copied 1 file (cross-volume or links unavailable). Downgrading policy to 'now' for those files because live updates won't propagate from the originals. Re-run wandb.save to resync, or place your run directory on the same drive to enable hardlinks.


[2026-06-08T17:32:42] test_evaluation_end: {"precision": 0.9362477444681241, "recall": 0.790636097613319, "mAP50": 0.8659065868805218, "mAP50-95": 0.6886184747156848, "F1": 0.8573029002267296, "speed_preprocess_ms": 1.1934238450657224, "speed_inference_ms": 1.5538986431571042, "speed_loss_ms": 0.00013713030836123437, "speed_postprocess_ms": 0.2614070464501051, "inference_time_ms": 1.5538986431571042}


...

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\fridge\_netrc


Partial benchmark results exported: E:\1. UM-Staff\AICOM\AICOM-OD\Results\YOLO26\Model-Size-Comparison\yolo26_multi_model_template_partial.csv
Consolidated benchmark summary exported: E:\1. UM-Staff\AICOM\AICOM-OD\Results\YOLO26\Model-Size-Comparison\yolo26_multi_model_template_summary.csv


summary_csv,E:\1. UM-Staff\AICOM...


,model_variant,parameter_count,model_size_mb,mAP@50,mAP@50-95,precision,recall,F1 Score,validation_inference_time_ms,test_inference_time_ms,...,test_mAP@50,test_mAP@50-95,test_precision,test_recall,test_F1 Score,best_weights,last_weights,run_name,status,wandb_url
0,yolo26n,2505750,5.14,0.869787,0.698370,0.922290,0.803722,0.858934,1.419915,1.553899,...,0.865907,0.688618,0.936248,0.790636,0.857303,E:\1. UM-Staff\AICOM\AICOM-OD\runs\YOLO26-EUVI...,E:\1. UM-Staff\AICOM\AICOM-OD\runs\YOLO26-EUVI...,E001_yolo26n_rfv20_img640_b32_e100_adamw_lr0p001,completed,https://wandb.ai/mark-bugeja-university-of-mal...
1,yolo26s,9951734,19.38,0.899887,0.755967,0.946972,0.848686,0.895139,2.544794,2.603902,...,0.901386,0.752661,0.948919,0.861911,0.903325,E:\1. UM-Staff\AICOM\AICOM-OD\runs\YOLO26-EUVI...,E:\1. UM-Staff\AICOM\AICOM-OD\runs\YOLO26-EUVI...,E002_yolo26s_rfv20_img640_b32_e100_adamw_lr0p001,completed,https://wandb.ai/mark-bugeja-university-of-mal...
2,yolo26m,21780598,42.00,0.907398,0.762111,0.940565,0.861405,0.899246,3.835848,3.713958,...,0.895147,0.759693,0.930651,0.862470,0.895264,E:\1. UM-Staff\AICOM\AICOM-OD\runs\YOLO26-EUVI...,E:\1. UM-Staff\AICOM\AICOM-OD\runs\YOLO26-EUVI...,E003_yolo26m_rfv20_img640_b32_e100_adamw_lr0p001,completed,https://wandb.ai/mark-bugeja-university-of-mal...
3,yolo26l,26184054,50.54,0.915210,0.795107,0.957815,0.873229,0.913568,4.690175,4.807778,...,0.897630,0.782444,0.971819,0.869772,0.917968,E:\1. UM-Staff\AICOM\AICOM-OD\runs\YOLO26-EUVI...,E:\1. UM-Staff\AICOM\AICOM-OD\runs\YOLO26-EUVI...,E004_yolo26l_rfv20_img640_b32_e100_adamw_lr0p001,completed,https://wandb.ai/mark-bugeja-university-of-mal...


: 